# 01 - USGS wildcat scoping: debris-flow hazard for the PROTECT Risk Index

**Purpose.** Debris Flow is on the PROTECT hazard list but no basin-wide hazard layer exists yet
(it is in the "needed, not created" column of the data inventory). This notebook scopes how we
would generate that layer with **USGS wildcat**, the agency's operational tool for post-fire
debris-flow hazard assessment. Nothing here runs a full assessment yet: the goal is to pin down
the environment, the input data mapping, the scenario design, and the skeleton workflow.

**What wildcat produces.** For a stream-segment network derived from a 10 m DEM, wildcat
estimates, per segment / outlet basin:

| Result | Units | Model |
|---|---|---|
| Debris-flow likelihood | 0-1 | M1 logistic model, [Staley et al. 2017](https://doi.org/10.1016/j.geomorph.2016.10.019) |
| Potential sediment volume | m3 | Emergency assessment model, [Gartner et al. 2014](https://doi.org/10.1016/j.enggeo.2014.04.008) |
| Combined hazard class | 1 low / 2 moderate / 3 high | Modified [Cannon et al. 2010](https://doi.org/10.1130/B26459.1) scheme |
| Rainfall thresholds | mm, mm/hr at 15/30/60 min | Inverted M1 model |

These map directly onto the Risk Index: likelihood and hazard class feed the **Exposure**
sub-index for the debris-flow hazard-asset pairs; rainfall thresholds connect to the climate
pipeline's precipitation-intensity work (Atlas 14 now, LOCA2/WRF-adjusted later); volumes
support consequence narratives (culvert/bridge blockage).

**Tool facts** (verified 2026-07-29 against [code.usgs.gov/ghsc/lhp/wildcat](https://code.usgs.gov/ghsc/lhp/wildcat)):

- wildcat **1.1.1**, Python >= 3.11 < 4, GPL-3.0. Docs: [ghsc.code-pages.usgs.gov/lhp/wildcat](https://ghsc.code-pages.usgs.gov/lhp/wildcat/)
- Built on **pfdf >= 3.0** ([docs](https://ghsc.code-pages.usgs.gov/lhp/pfdf/)), the USGS post-fire
  debris-flow Python library. wildcat = turnkey USGS-style assessments; pfdf = the building blocks
  (and data acquisition helpers) if we need to customize.
- Four commands, CLI or Python API: `initialize` -> `preprocess` -> `assess` -> `export`.
- Installs from the **USGS GitLab package registry**, not pypi.org (the PyPI `wildcat` is an
  unrelated quantum SDK). See `environment.md`.
- Depends on `rasterio` + `fiona`: **must live in its own env, never arcgispro-py3**.

## 1. Environment check

This notebook should run on the dedicated `wildcat` kernel (see `environment.md` for the two
commands that create and register it). The cell below reports what it finds and does not fail
if wildcat is not installed yet: the config/mapping/design-storm sections only need
pandas + pyyaml, so they also run on arcgispro-py3 in the meantime.

In [ ]:
import sys
print(f"Interpreter: {sys.executable}")
print(f"Python:      {sys.version.split()[0]}")

WILDCAT_OK = False
try:
    import wildcat, pfdf
    from importlib.metadata import version
    WILDCAT_OK = True
    print(f"wildcat:     {version('wildcat')}")
    print(f"pfdf:        {version('pfdf')}")
except ImportError as e:
    print(f"wildcat not available in this kernel ({e}).")
    print("Scoping cells still run; assessment cells are skipped. See environment.md.")

In [ ]:
# Load pipeline config + logger (paths relative to debris-flow/)
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.io import load_config, get_logger

cfg = load_config()
log = get_logger("01_wildcat_scoping")
log.info(f"Project: {cfg['project']['name']}")
log.info(f"wildcat target version: {cfg['wildcat']['version']}")

## 2. How a wildcat assessment runs

Each assessment is a self-contained **project folder** (we keep them under
`debris-flow/projects/`, one per scenario, all gitignored):

```
projects/<scenario>/
├── configuration.py     # all settings; created by initialize, edited by us
├── inputs/              # or absolute paths to data/raw/ for large rasters
├── preprocessed/        # preprocess: everything reprojected/clipped to the DEM grid
├── assessment/          # assess: segments / outlets / basins .geojson + config record
└── exports/             # export: Shapefile/GeoJSON in a chosen CRS, formatted fields
```

Key behaviors that shape our design:

- **The fire perimeter defines the domain.** The perimeter is buffered (default 3 km) and every
  raster is clipped to that extent. For the pre-fire planning scenario we must therefore supply a
  hypothetical "perimeter" (see section 4).
- **The DEM sets the grid.** All inputs are reprojected to the DEM's CRS/resolution/alignment.
  The models are calibrated for ~10 m DEMs; wildcat enforces 6.5-11 m by default.
- **dNBR and severity can be constants.** `dnbr = 525` or `severity = 3` in `configuration.py`
  applies a uniform value across the watershed: this is the hook for hypothetical-burn scenarios.
- **Every step writes a `configuration.txt` record** that exactly reproduces the run: good
  provenance for the RIP appendix.
- **Network filtering is tunable** (`min_area_km2`, `min_slope`, `max_confinement`,
  `max_exterior_ratio`, ...). Defaults implement the USGS operational style; we start there.
- Python API mirrors the CLI and accepts kwargs that override `configuration.py`, which is how
  we will sweep parameters from a notebook.

## 3. Input datasets mapped to Tahoe sources

wildcat's standard inputs and where we would get each one for the basin. pfdf ships data
acquisition helpers (`pfdf.data`) for several of these; module paths to be confirmed after
install.

In [ ]:
import pandas as pd

mapping = pd.DataFrame([
    # dataset, required, tahoe_source, status, notes
    ["perimeter", "required", "Scenario-dependent: real fire perimeter (NIFC/CAL FIRE) or hypothetical unit (HUC12s / TRPA boundary)", "decision open", "Buffered by buffer_km; defines the analysis domain"],
    ["dem", "required", "USGS 3DEP 1/3 arc-second (~10 m); pfdf.data helper or TNM download", "not acquired", "Sets CRS/resolution/alignment; keep 10 m, do not substitute lidar 1 m"],
    ["dnbr", "recommended", "Validation: MTBS/RAVG (Caldor 2021, Angora 2007). Pre-fire: constant value", "not acquired", "Expected scaling raw dNBR * 1000"],
    ["severity", "recommended", "BARC4 from BAER where available; else estimated from dNBR; pre-fire: constant class", "not acquired", "BARC4 classes 1-4"],
    ["kf", "recommended", "STATSGO KF-factor (pfdf.data helper); attribute field KFFACT", "not acquired", "Polygon file needs kf_field setting"],
    ["evt", "recommended", "LANDFIRE Existing Vegetation Type, latest", "not acquired", "Builds water / developed / excluded masks"],
    ["retainments", "optional", "No basin-wide debris-basin inventory known; ask jurisdictions about sediment basins", "decision open", "Pixels downstream of retainments excluded from network"],
    ["iswater", "optional", "Lake Tahoe + major lakes; EVT water mask likely sufficient, else TRPA hydro layers", "probably skip", "EVT-derived mask is the default path"],
    ["isdeveloped", "optional", "EVT development mask; TRPA impervious/urban layers if EVT is too coarse", "probably skip", "Informs network filtering"],
], columns=["dataset", "tier", "tahoe_source", "status", "notes"])

out_csv = ROOT / cfg["paths"]["outputs"] / "wildcat_input_mapping.csv"
out_csv.parent.mkdir(exist_ok=True)
mapping.to_csv(out_csv, index=False)
log.info(f"Input mapping written: {out_csv} ({len(mapping)} datasets)")
mapping

## 4. Scenario design

wildcat is a *post*-fire tool: it expects a fire that already happened. PROTECT needs a
*planning-scale* hazard layer for roads that have not burned yet. USGS handles this with
**pre-fire assessments**: run the same models over hypothetical burns (e.g.,
[Staley et al. 2018, SIR 2016-5148](https://doi.org/10.3133/sir20165148) style work), which
wildcat supports directly through constant-valued `dnbr` / `severity`.

Three scenarios, in build order:

**S1 - Validation (real fire).** Run wildcat on the Caldor 2021 footprint (burned into the
basin at Echo Summit / Christmas Valley) with real dNBR, USGS defaults. USGS published an
operational Caldor assessment, so we can sanity-check our outputs against theirs. Angora 2007
is the smaller backup candidate.

**S2 - Pre-fire planning (the deliverable).** Hypothetical burn at constant moderate/high
severity over watersheds draining to the transportation network. Open sub-decisions:

- *Burn unit*: HUC12 watersheds intersecting highway corridors (closer to how USGS frames
  pre-fire runs, keeps domains small) vs. one basin-wide run (simpler, but a ~50 km domain at
  10 m needs a feasibility check, especially basin delineation).
- *Severity scenario*: constant severity 3 everywhere is the blunt v1; a defensible v2 uses
  modeled severity (e.g., flame-length-derived classes, as USGS pre-fire work does). The
  Vegetation_Burn_Severity / Forest_Health services and WEPP inputs may already carry usable
  modeled-severity surfaces.
- *dNBR constant*: pick a value consistent with the severity class (placeholder 525 in
  config.yaml) and document the choice.

**S3 - Climate-adjusted storms.** Re-run S2 swapping the design-storm `I15_mm_hr` values:
current climate from the Atlas 14 15-minute DDF (already extracted by the climate pipeline),
future climate once the LOCA2/WRF sub-daily intensity scaling lands. This gives the Risk Index
a hazard layer that responds to the climate horizons (2020-2049 / 2040-2069 / 2070-2099).

**Relationship to WEPP and RAMMS.** WEPP gives post-fire erosion/sediment yield (chronic
loading); wildcat gives event debris-flow probability/volume on the channel network; RAMMS
(sole-source) simulates runout for specific sites. They are complementary: wildcat is the
screening layer for the index, RAMMS a possible follow-up on hotspots wildcat flags.

## 5. Design storm: I15 from the climate pipeline

The M1 model's rainfall input is peak 15-minute intensity. wildcat's USGS-default set is
`[16, 20, 24, 40]` mm/hr. The cell below pulls the basin's actual Atlas 14 15-minute
intensities from the climate pipeline output so we can choose defensible, Tahoe-specific
values (and later swap in climate-adjusted ones).

In [ ]:
ddf_path = (ROOT / cfg["design_storm"]["atlas14_ddf_csv"]).resolve()

if ddf_path.exists():
    ddf = pd.read_csv(ddf_path)
    d15 = ddf[ddf["duration"] == cfg["design_storm"]["duration"]].copy()
    d15["intensity_mm_hr"] = d15["depth_mm"] * 4  # 15-min depth -> hourly rate
    i15 = (d15.pivot_table(index="ari_years", columns="point",
                           values="intensity_mm_hr")
              .round(1))
    i15["basin_median"] = i15.median(axis=1).round(1)
    log.info(f"Atlas 14 15-min intensities loaded from {ddf_path.name}: "
             f"{len(d15)} rows, {d15['point'].nunique()} points")
    display(i15)
    sel = i15.loc[i15.index.isin(cfg["design_storm"]["ari_years"]), "basin_median"]
    print(f"Candidate I15_mm_hr (basin median at ARIs {cfg['design_storm']['ari_years']}): "
          f"{sel.tolist()}")
    print(f"USGS default set for comparison: {cfg['design_storm']['i15_default_mm_hr']}")
else:
    log.warning(f"Atlas 14 DDF not found at {ddf_path} - run the climate pipeline first")

## 6. Skeleton workflow (guarded - does not run until data is in place)

The full run per scenario, using the Python API with kwargs overriding `configuration.py`.
Flip `RUN_WILDCAT = True` once the env exists and the S1 inputs are in `data/raw/`.

In [ ]:
RUN_WILDCAT = False   # flip deliberately
SCENARIO = "caldor_validation"

if RUN_WILDCAT and WILDCAT_OK:
    from wildcat import initialize, preprocess, assess, export

    project = ROOT / cfg["paths"]["projects"] / SCENARIO

    # 1) Create the project folder + default configuration.py (USGS-style defaults).
    #    Use config="full" to expose every setting for reference.
    if not project.exists():
        initialize(project=project)
        log.info(f"Initialized wildcat project: {project}")

    # 2) Point configuration.py at the inputs (edit the file, or override via kwargs
    #    as sketched here). Large rasters stay in data/raw/ - absolute paths are fine.
    raw = ROOT / cfg["paths"]["raw"]
    preprocess(
        project,
        perimeter=raw / "caldor_perimeter.shp",
        dem=raw / "dem_10m.tif",
        dnbr=raw / "caldor_dnbr.tif",
        # severity omitted -> estimated from dNBR
        kf=raw / "statsgo_kf.shp", kf_field="KFFACT",
        evt=raw / "landfire_evt.tif",
    )

    # 3) Assess with Tahoe design storms (section 5). locate_basins is the slow step;
    #    disable it on parameter sweeps, enable for deliverable runs.
    assess(
        project,
        I15_mm_hr=cfg["design_storm"]["i15_default_mm_hr"],  # swap for Atlas 14 picks
        locate_basins=True,
    )

    # 4) Export: GeoJSON in UTM 10N for ETL, plus the web CRS for the map tool.
    export(project, format="GeoJSON", crs=cfg["crs"]["target_epsg"])
    log.info(f"Assessment complete: {project / 'exports'}")
else:
    log.info("RUN_WILDCAT is False (or wildcat missing) - skeleton not executed")

## 7. Post-processing sketch: from wildcat output to the Risk Index

Once a scenario run exists, the export feeds the index roughly like this (stub - becomes
notebook 02 when real outputs exist):

1. Read `exports/segments.geojson` + `basins.geojson` with geopandas (EPSG:26910).
2. Intersect basins (and a buffered segment corridor) with the road network
   (Transportation service / the Overture-derived street network built for the
   disruption-risk models) and culvert/bridge assets.
3. Per asset: max combined hazard class, max likelihood at the chosen design storm, summed
   upstream potential volume -> the debris-flow **Exposure** score inputs.
4. Write a review CSV to `outputs/` and, once stable, load the hazard layer toward the
   packaged **Hazards** REST service and the `html/data-model-inventory.html` tracker.

## 8. Open decisions

1. Create the `wildcat` conda env (commands in `environment.md`) - needs a go-ahead since it
   is a new environment + package installs.
2. S1 validation fire: Caldor 2021 (bigger, has an official USGS assessment to compare
   against) vs. Angora 2007 (fully inside the basin, smaller). Leaning Caldor.
3. S2 burn unit: HUC12s intersecting corridors vs. basin-wide single run (feasibility check
   needed at 10 m).
4. S2 severity scenario: constant class 3 (v1) vs. modeled severity surface (v2); pick the
   dNBR constant to match.
5. Design-storm ARIs for the index (config placeholder: 2/10/25-yr basin median) and how S3
   future scaling gets derived from the climate pipeline.
6. Whether any debris retainment / sediment basin inventory exists to include.
7. How wildcat outputs are versioned into the Hazards REST service and the data inventory
   page (readiness status bump for Debris Flow).

**Sources:** [wildcat docs](https://ghsc.code-pages.usgs.gov/lhp/wildcat/) | [wildcat repo](https://code.usgs.gov/ghsc/lhp/wildcat) | [pfdf docs](https://ghsc.code-pages.usgs.gov/lhp/pfdf/) | [Staley et al. 2017](https://doi.org/10.1016/j.geomorph.2016.10.019) | [Gartner et al. 2014](https://doi.org/10.1016/j.enggeo.2014.04.008) | [Cannon et al. 2010](https://doi.org/10.1130/B26459.1) | [USGS PFDF hazard dashboard](https://www.arcgis.com/apps/dashboards/c09fa874362e48a9afe79432f2efe6fe)